In [1]:
import sys
sys.executable

'/workspaces/LLM_workspace/02_vector_search/.venv/bin/python'

# Module 2 Homework: Vector Search

Link to the github repo: [Link](https://github.com/DataTalksClub/llm-zoomcamp/tree/main)

## Q1. Embedding a query

Embed the following query:

In [2]:
from embedder import Embedder
query = "How does approximate nearest neighbor search work?"
embed = Embedder()
q1 = embed.encode(query)
q1[0]

2026-06-22 21:04:39.354074169 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


np.float64(-0.02058203437252893)

## Loading the data

Let's pull the lesson pages from the repository:

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
len(documents)
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Cosine similarity

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its content, and compute the cosine similarity with the query vector from Q1.

In [5]:
page = "02-vector-search/lessons/07-sqlitesearch-vector.md"
document = next((doc for doc in documents if doc["filename"] == page), None)

doc_encode = embed.encode(document["content"])
doc_encode.dot(q1)

np.float64(0.36107027225589694)

## Q3. Chunking and search by hand
Let's chunk the documents to split the topics at each document and then find the highest similarity chunk with q1

In [6]:
from gitsource import chunk_documents
from tqdm import tqdm
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

X = []
for i in tqdm(range(len(chunks))):
    chunk = chunks[i]
    batch_encode = embed.encode(chunk["content"])
    X.append(batch_encode)

X = np.array(X)

scores = X.dot(q1)
idx = np.argmax(scores)
chunks[idx]

  0%|          | 0/295 [00:00<?, ?it/s]

100%|██████████| 295/295 [00:19<00:00, 15.02it/s]


{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q4. Vector search with minsearch

In [7]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)


query = 'What metric do we use to evaluate a search engine?'
q2 = embed.encode(query)
results = vindex.search(q2, num_results=1)

results[0]


{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

## Q5. Text search vs vector search

Let's compare the text search with vector search:

In [11]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)

index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"
q3 = embed.encode(query)

r_vs = vindex.search(q3, num_results=5)
r_ts = index.search(query, num_results=5)

r_vs_filenames = [r["filename"] for r in r_vs]
r_ts_filenames = [r["filename"] for r in r_ts]
print("Vector search results:", r_vs_filenames)
print("Text search results:", r_ts_filenames)


Vector search results: ['02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/08-pgvector.md', '02-vector-search/lessons/08-pgvector.md']
Text search results: ['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md']


## Q6. Hybrid search

In [12]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query = "How do I give the model access to tools?"
q4 = embed.encode(query)

r_vs = vindex.search(q4, num_results=5)
r_ts = index.search(query, num_results=5)

results = rrf([r_vs, r_ts])

results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'